In [1]:
!pip install -q kagglehub

In [2]:
import os
import re
import json
import unicodedata
from collections import Counter

import pandas as pd
from google.colab import drive


In [3]:
drive.mount('/content/drive')

PASTA = '/content/drive/MyDrive/Colab Notebooks/Card 27/Pratica/dadosLLM'  # pasta onde os arquivos serão salvos
os.makedirs(PASTA, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
SECAO = 'esporte'               # seção usada no fine-tuning
LIMITE_GERAL = 30_000_000       # caracteres do pré-treino
LIMITE_SECAO = 8_000_000        # caracteres do fine-tuning

In [5]:
LETRAS = 'abcdefghijklmnopqrstuvwxyz'
ACENTOS = 'áàâãéêíóôõúüç'
VOCAB = sorted(set(
    LETRAS + LETRAS.upper() + ACENTOS + ACENTOS.upper()
    + '0123456789'
    + ' \n.,;:!?()[]-"\'/%$&+=*°ºª'
))
print(len(VOCAB), 'caracteres')
print(repr(''.join(VOCAB)))

# símbolos parecidos viram a versão simples
TROCAS = str.maketrans({
    '“': '"', '”': '"', '„': '"', '«': '"', '»': '"',
    '‘': "'", '’': "'", '´': "'", '`': "'",
    '–': '-', '—': '-', '−': '-', '‐': '-',
    '…': '...', '\u00a0': ' ', '\t': ' ',
})

# tudo que não está no vocabulário
FORA_DO_VOCAB = re.compile('[^' + re.escape(''.join(VOCAB)) + ']')


def normalizar(texto):
    # junta letra + acento num caractere só
    texto = unicodedata.normalize('NFC', str(texto))
    texto = texto.translate(TROCAS)
    return FORA_DO_VOCAB.sub('', texto)

113 caracteres
'\n !"$%&\'()*+,-./0123456789:;=?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]abcdefghijklmnopqrstuvwxyzª°ºÀÁÂÃÇÉÊÍÓÔÕÚÜàáâãçéêíóôõúü'


## Notícias da Folha

Base do Kaggle *News of the Brazilian Newspaper*, com cerca de 167 mil notícias (título, texto, data e categoria).


In [6]:
CSV = os.path.join(PASTA, 'articles.csv')
if not os.path.exists(CSV):
    import kagglehub
    caminho = kagglehub.dataset_download('marlesson/news-of-the-site-folhauol')
    CSV = os.path.join(caminho, 'articles.csv')

noticias = pd.read_csv(CSV)
print(noticias.shape)
print(noticias.columns.tolist())
noticias.head()

(167053, 6)
['title', 'text', 'date', 'category', 'subcategory', 'link']


,title,text,date,category,subcategory,link
0,"Lula diz que está 'lascado', mas que ainda tem...",Com a possibilidade de uma condenação impedir ...,2017-09-10,poder,NaN,http://www1.folha.uol.com.br/poder/2017/10/192...
1,"'Decidi ser escrava das mulheres que sofrem', ...","Para Oumou Sangaré, cantora e ativista malines...",2017-09-10,ilustrada,NaN,http://www1.folha.uol.com.br/ilustrada/2017/10...
2,Três reportagens da Folha ganham Prêmio Petrob...,Três reportagens da Folha foram vencedoras do ...,2017-09-10,poder,NaN,http://www1.folha.uol.com.br/poder/2017/10/192...
3,Filme 'Star Wars: Os Últimos Jedi' ganha trail...,A Disney divulgou na noite desta segunda-feira...,2017-09-10,ilustrada,NaN,http://www1.folha.uol.com.br/ilustrada/2017/10...
4,CBSS inicia acordos com fintechs e quer 30% do...,"O CBSS, banco da holding Elopar dos sócios Bra...",2017-09-10,mercado,NaN,http://www1.folha.uol.com.br/mercado/2017/10/1...


In [7]:
noticias = noticias.dropna(subset=['title', 'text', 'category'])
noticias = noticias.drop_duplicates(subset='text')
noticias = noticias.sample(frac=1, random_state=42)  # embaralha para misturar seções e anos

print(len(noticias), 'notícias depois de remover vazias e duplicadas')
noticias['category'].value_counts().head(15)

165901 notícias depois de remover vazias e duplicadas


,count
category,
poder,22005
colunas,21499
mercado,20936
esporte,19723
mundo,17123
cotidiano,16952
ilustrada,15604
opiniao,4523
paineldoleitor,4008


In [8]:
def limpar(coluna):
    coluna = coluna.map(normalizar)
    coluna = coluna.str.replace(r'[ \t]+', ' ', regex=True)          # espaços repetidos
    coluna = coluna.str.replace(r' *\n *', '\n', regex=True)          # espaços no começo/fim das linhas
    coluna = coluna.str.replace(r'\n{3,}', '\n\n', regex=True)        # no máximo uma linha em branco
    coluna = coluna.str.replace(r' +([,.;:!?])', r'\1', regex=True)   # espaço antes de pontuação
    return coluna.str.strip()


noticias['titulo'] = limpar(noticias['title'])
noticias['texto'] = limpar(noticias['text'])
noticias['secao'] = limpar(noticias['category']).str.lower().str.replace(' ', '-', regex=False)

noticias = noticias[(noticias['texto'].str.len() >= 300) & (noticias['titulo'].str.len() > 0)]
noticias['bloco'] = '[' + noticias['secao'] + '] ' + noticias['titulo'] + '\n' + noticias['texto'] + '\n\n'

print(len(noticias), 'notícias depois da limpeza')
print(noticias['bloco'].iloc[0][:600])

163073 notícias depois da limpeza
[opiniao] Os desafios para o futebol durante a crise
O futebol brasileiro vive um momento de profundas mudanças. Em meio à grave crise econômica que o país atravessa, os clubes precisarão praticar uma nova forma de gestão. A aprovação do Profut (Programa de Modernização da Gestão e de Responsabilidade Fiscal do Futebol Brasileiro) no Congresso, apesar de positiva para auxiliar na equação dos problemas com o pagamento de impostos atrasados, fará com que os times tenham que adotar uma rigorosa disciplina no uso de seus recursos para a realização de custos operacionais e investimentos. Dentro des


In [9]:
assert SECAO in set(noticias['secao']), f'Seção {SECAO!r} não encontrada. Opções: {noticias["secao"].value_counts().index[:15].tolist()}'

geral = noticias[noticias['secao'] != SECAO]
secao = noticias[noticias['secao'] == SECAO]


def salvar(df, nome, limite):
    # mantém notícias até atingir o limite de caracteres e grava num .txt
    tamanho = df['bloco'].str.len()
    df = df[tamanho.cumsum() <= limite]
    with open(os.path.join(PASTA, nome), 'w', encoding='utf-8') as f:
        f.write(''.join(df['bloco']))
    print(f'{nome}: {len(df)} notícias, {df["bloco"].str.len().sum() / 1e6:.1f}M caracteres')
    return df


geral = salvar(geral, 'folha_geral.txt', LIMITE_GERAL)
secao = salvar(secao, 'folha_secao.txt', LIMITE_SECAO)

geral['secao'].value_counts().head(10)

folha_geral.txt: 10148 notícias, 30.0M caracteres
folha_secao.txt: 3579 notícias, 8.0M caracteres


,count
secao,
poder,1532
mercado,1472
colunas,1435
mundo,1185
cotidiano,1178
ilustrada,1121
paineldoleitor,293
opiniao,292
saopaulo,284


O vocabulário é salvo em `vocab.json` para o notebook de treino usar exatamente o mesmo mapeamento. No fim, confere se os arquivos só têm caracteres do vocabulário.

In [10]:
with open(os.path.join(PASTA, 'vocab.json'), 'w', encoding='utf-8') as f:
    json.dump({'chars': VOCAB, 'secao': SECAO}, f, ensure_ascii=False)

for nome in ['folha_geral.txt', 'folha_secao.txt']:
    with open(os.path.join(PASTA, nome), 'r', encoding='utf-8') as f:
        texto = f.read()
    extras = set(texto) - set(VOCAB)
    print(f'{nome}: {len(texto) / 1e6:.1f}M caracteres, fora do vocabulário: {extras or "nenhum"}')
    print(texto[:500])
    print('-' * 80)

folha_geral.txt: 30.0M caracteres, fora do vocabulário: nenhum
[opiniao] Os desafios para o futebol durante a crise
O futebol brasileiro vive um momento de profundas mudanças. Em meio à grave crise econômica que o país atravessa, os clubes precisarão praticar uma nova forma de gestão. A aprovação do Profut (Programa de Modernização da Gestão e de Responsabilidade Fiscal do Futebol Brasileiro) no Congresso, apesar de positiva para auxiliar na equação dos problemas com o pagamento de impostos atrasados, fará com que os times tenham que adotar uma rigorosa dis
--------------------------------------------------------------------------------
folha_secao.txt: 8.0M caracteres, fora do vocabulário: nenhum
[esporte] Leicester se classifica e Dortmund iguala recorde da Liga dos Campeões
Dono do título inglês da temporada passada, o Leicester venceu mais uma vez na Liga dos Campeões, nesta terça-feira (22), ao emplacar 2 a 1 em casa no belga Club Bruges, em partida válida pela quinta rodada do Gr